# CFPB Complaint Ingestion

Pulls Truist, JPMorgan Chase, and American Express complaints from the CFPB Complaint Search API, year by year (2020-2026), and writes the cleaned result to the `staging_lakehouse` as `complaints_2026_clean`.

Pulled year-by-year rather than in one request because JPMorgan Chase alone has 100K+ complaints in scope, enough to hit the API's result-size cap when requested in a single shot. Uses a browser-style `User-Agent` header because CFPB's site blocks Python's default request identity (403). The `consumer_complaint_narrative` field is dropped before it ever reaches the lakehouse -- out of scope for this project, and the single most error-prone field for parsing corruption in earlier ingestion attempts.

In [ ]:
import pandas as pd
import requests
from io import StringIO
from urllib.parse import quote

# --- Config ---
base_url = "https://www.consumerfinance.gov/data-research/consumer-complaints/search/api/v1/"

companies = [
    "TRUIST FINANCIAL CORPORATION",
    "JPMORGAN CHASE & CO.",
    "AMERICAN EXPRESS COMPANY",
]

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept": "text/csv,*/*",
}

def enc(s):
    return quote(s, safe='')

# --- Fetch year by year (avoids the API's result-size cap on large pulls) ---
frames = []
for year in range(2020, 2027):
    date_min = f"{year}-01-01"
    date_max = f"{year}-12-31"
    query = "format=csv&" + "&".join(f"company={enc(c)}" for c in companies)
    query += f"&date_received_min={date_min}&date_received_max={date_max}"
    url = base_url + "?" + query

    resp = requests.get(url, headers=headers, timeout=120)
    if resp.status_code != 200:
        print(f"YEAR {year} FAILED: {resp.status_code}")
        print(resp.text[:1000])
        continue

    year_df = pd.read_csv(StringIO(resp.text))
    print(f"{year}: {year_df.shape[0]} rows")
    frames.append(year_df)

df = pd.concat(frames, ignore_index=True)

# --- Rename to snake_case to match the warehouse schema ---
df = df.rename(columns={
    "Date received": "date_received",
    "Product": "product",
    "Sub-product": "sub_product",
    "Issue": "issue",
    "Sub-issue": "sub_issue",
    "Consumer complaint narrative": "consumer_complaint_narrative",
    "Company public response": "company_public_response",
    "Company": "company",
    "State": "state",
    "ZIP code": "zip_code",
    "Tags": "tags",
    "Submitted via": "submitted_via",
    "Date sent to company": "date_sent_to_company",
    "Company response to consumer": "company_response_to_consumer",
    "Timely response?": "timely_response",
    "Complaint ID": "complaint_id",
})

# Drop narrative text -- out of scope for this project, and the field most prone to breaking CSV parsers
df = df.drop(columns=["consumer_complaint_narrative"], errors="ignore")

# --- Write to the lakehouse (requires staging_lakehouse attached as the notebook's default) ---
spark_df = spark.createDataFrame(df.astype(str))
spark_df.write.mode("overwrite").saveAsTable("complaints_2026_clean")

# --- Verify ---
print("TOTAL:", df.shape)
print(df['company'].value_counts(dropna=False))
print(df['product'].value_counts(dropna=False).head(10))